# 01 · Tu galaxia de cumpleaños con la API de la NASA

En este cuaderno vas a hacer tu **primera petición a una API** de verdad: le vas a pedir a la NASA **fotos de galaxias** del año en que naciste, vas a recibir "tu galaxia de cumpleaños" y la vas a **descargar** a tu computador.

### Solo tienes que:
1. Ejecutar las celdas **en orden** (botón ▶ o `Shift + Enter`).
2. Poner **tu fecha de cumpleaños** cuando te la pida.
3. ¡Descargar tu galaxia! 

---
## Antes de empezar: ¿qué es una API?

Piensa en un **restaurante**:

| Restaurante | API |
|---|---|
| Tú pides la comida | Tu código hace una **petición** |
| El mesero lleva el pedido | La **API** lleva tu petición |
| La cocina prepara el plato | El **servidor de la NASA** busca las fotos |
| Te traen el plato | Recibes una **respuesta** con los datos |

Para que el mesero te entienda, el pedido tiene 2 partes:

-  **La dirección (URL):** a dónde le pides → `https://images-api.nasa.gov/search`
-  **Los parámetros:** los detalles del pedido → *"fotos de **galaxias** del año **2005**"*

Usaremos la **NASA Image and Video Library**: la biblioteca pública de la NASA, con más de 140.000 fotos del espacio, de telescopios como el **Hubble** y el **James Webb**.

---
## Paso 1: Preparar las herramientas

`requests` es la librería de Python para hacer peticiones a internet.

In [12]:
import requests
from IPython.display import Image, display

print(" Listo, herramientas cargadas")

 Listo, herramientas cargadas


---
## Paso 2: Escribe tu cumpleaños y elige qué buscar

1. Usa el **selector de fecha** de la derecha para elegir tu cumpleaños.
2. En **qué_buscar** puedes dejar `galaxy` (galaxia) o elegir otra cosa del espacio.
3. Ejecuta la celda.

In [ ]:
mi_cumpleaños = "2000-01-01"  #@param {type:"date"}
qué_buscar = "galaxy"  #@param ["galaxy", "nebula", "black hole", "supernova", "star cluster"]

año = mi_cumpleaños[:4]   # los primeros 4 caracteres de la fecha = el año
print(f" Vamos a buscar fotos de '{qué_buscar}' del año {año}")

**texto en negrita**>  Las búsquedas van en **inglés** porque así están guardadas las fotos en la NASA: `galaxy` = galaxia, `nebula` = nebulosa, `black hole` = agujero negro, `supernova` = supernova, `star cluster` = cúmulo de estrellas.

---
## Paso 3: Hacer la petición a la NASA

Aquí ocurre la magia. Fíjate en las partes del pedido:

In [ ]:
# 1. La dirección
url = "https://images-api.nasa.gov/search"

# 2. Los detalles del pedido
parametros = {
    "q": qué_buscar,          # q = "query" = lo que se buscara en la base de datos de la NASA
    "media_type": "image",    # sE Generan fotos
    "year_start": año,        # desde el año...
    "year_end": año,          # ...hasta el mismo año
}

# 3. ¡Hacemos la petición!
respuesta = requests.get(url, params=parametros)

print("Código de respuesta:", respuesta.status_code)

### ¿Qué significa ese número "Código de Respuesta?

| Código | Significado |
|---|---|
| **200**  | ¡Todo salió bien! Sigue al paso 4 |
| **400**  | El pedido está mal escrito. Revisa el paso 2 |
| **404**  | La dirección no existe. Revisa la URL |
| **500**  | El servidor de la NASA tiene problemas. Intenta en un minuto |

 **Truco:** si empieza por **2**, todo bien. Si empieza por **4**, hay que revisar algo de nuestro lado. Si empieza por **5**, el problema es del servidor.

---
## Paso 4: Abrir la respuesta

La NASA responde en un formato llamado **JSON**. En Python funciona como un **diccionario** (datos con nombre y valor) que guarda **listas** adentro, como cajas dentro de cajas.

La lista de fotos está en `datos["collection"]["items"]`:

In [ ]:
datos = respuesta.json()
fotos = datos["collection"]["items"]

print(f" La NASA encontró {len(fotos)} fotos de '{qué_buscar}' del año {año}")

# Si no hay fotos de ese año, buscamos en todos los años
if len(fotos) == 0:
    print(" No hay fotos de ese año. Busquemos en todos los años...")
    del parametros["year_start"]
    del parametros["year_end"]
    respuesta = requests.get(url, params=parametros)
    fotos = respuesta.json()["collection"]["items"]
    print(f" Ahora encontramos {len(fotos)} fotos")

Cada foto trae su **información** (título, fecha, descripción) y sus **enlaces** a la imagen. Veamos los títulos de las primeras 5:

In [ ]:
for foto in fotos[:5]:
    info = foto["data"][0]
    print("•", info["title"])

---
## Paso 5: ¡Tu galaxia de cumpleaños!

Ahora elegimos **una** foto usando tu día y mes de nacimiento. Así cada persona recibe una galaxia distinta (y si lo vuelves a ejecutar con la misma fecha, siempre te sale la misma).

In [ ]:
mes = int(mi_cumpleaños[5:7])
dia = int(mi_cumpleaños[8:10])

# Una "fórmula" con tu día y mes para escoger una foto de la lista
posición = (dia * 31 + mes) % len(fotos)
mi_foto = fotos[posición]

info = mi_foto["data"][0]
vista_previa = mi_foto["links"][0]["href"]

print(" Título:", info["title"])
print(" Fecha en la NASA:", info["date_created"][:10])
print(" Centro de la NASA:", info.get("center", "No indicado"))
print()
display(Image(url=vista_previa, width=600))

### ¿Qué es lo que ves?

La NASA escribe una descripción de cada foto.

In [ ]:
print(info.get("description", "Sin descripción disponible"))

---
## Paso 6: ¡Descarga tu galaxia en alta calidad! 

La foto que vimos es una vista previa pequeña. Para la versión grande hacemos **una segunda petición**, a otra dirección de la API: `/asset/`, que nos da todos los tamaños disponibles de esa foto.

In [ ]:
# Cada foto tiene un código único: el nasa_id
nasa_id = info["nasa_id"]
print(" Código de la foto:", nasa_id)

# Segunda petición: pedimos los archivos de esa foto
respuesta_archivos = requests.get(f"https://images-api.nasa.gov/asset/{nasa_id}")
enlaces = [archivo["href"] for archivo in respuesta_archivos.json()["collection"]["items"]]

# Elegimos el mejor tamaño disponible: grande → mediano → original
enlace_imagen = vista_previa
for tamaño in ["~large.jpg", "~medium.jpg", "~orig.jpg"]:
    encontrados = [e for e in enlaces if e.endswith(tamaño)]
    if encontrados:
        enlace_imagen = encontrados[0]
        break

enlace_imagen = enlace_imagen.replace("http://", "https://")
print(" Imagen elegida:", enlace_imagen)

In [ ]:
# Tercera petición: ahora sí pedimos la imagen
imagen = requests.get(enlace_imagen)

# La guardamos con un nombre que incluye tu cumpleaños
nombre_archivo = f"mi_galaxia_{mi_cumpleaños}.jpg"
with open(nombre_archivo, "wb") as archivo:
    archivo.write(imagen.content)

print(" Imagen guardada como:", nombre_archivo)

# La descargamos a tu computador
try:
    from google.colab import files
    files.download(nombre_archivo)
    print(" ¡Descarga iniciada! Revisa tu carpeta de Descargas.")
except ImportError:
    print("No estás en Colab: el archivo quedó en la carpeta del cuaderno.")

> ** Si tu navegador pregunta si permites descargas de Colab, haz clic en **

**Permitir**.

---
## ¡Lo lograste!

Acabas de hacer **3 peticiones** reales a una API. Repasemos:

| Hiciste esto | Se llama |
|---|---|
| Le pediste fotos a la NASA | **Petición** (*request*) |
| Usaste la dirección `images-api.nasa.gov/search` | **URL / endpoint** |
| Enviaste qué buscar y el año | **Parámetros** |
| Recibiste un 200 | **Código de estado** |
| Sacaste la lista con `datos["collection"]["items"]` | Leer el **JSON** |
| Usaste `/asset/` para la foto grande | Un **segundo endpoint** de la misma API |
| Bajaste la imagen | Una petición por un **archivo** |

---
## Retos (opcionales)

1.  **Para tu familia:** vuelve al paso 2, pon el cumpleaños de alguien de tu familia y descarga su galaxia.
2.  **Cambia de tema:** en el paso 2 elige `black hole` o `nebula` y ejecuta todo de nuevo.
3.  **Busca tu propio tema:** en el paso 3 cambia `qué_buscar` por algo escrito por ti, como `"Mars"`, `"Saturn"` o `"Hubble"`.
4.  **Cuenta:** ¿cuántas fotos de galaxias hay de tu año y cuántas del año 2023? ¿Por qué crees que cambia tanto?

In [ ]:
# Espacio para tus retos 